In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# CHEBYSHEV TYPE I LOW-PASS FILTER DESIGN EXERCISE
#
# Specifications:
#
#       ωp = 1.0 rad/s
#       ωs = 2.2 rad/s
#       Ap = 0.2 dB
#       As = 40.0 dB
#
# Philosophy:
#
# 1. N, ε and α are computed numerically from the theoretical equations.
# 2. The stable poles are computed numerically from the Chebyshev-I pole
#    equations.
# 3. No pole or final coefficient is hard-coded.
# 4. SymPy then constructs:
#
#       pole factors
#       H(s)
#       H(jω)
#       |H(jω)|
#       phase response
#       group delay
#
# This avoids extremely heavy exact symbolic manipulation of expressions
# containing irrational powers, inverse hyperbolic functions and radicals.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# FILTER SPECIFICATIONS
# ==============================================================================

wp = 1.0
ws = 2.2
Ap = 0.2
As = 40.0

# ==============================================================================
# STEP 1: MINIMUM FILTER ORDER
# ==============================================================================

epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

N_exact = np.arccosh(np.sqrt((10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0))) / np.arccosh(ws / wp)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 2: AUXILIARY CHEBYSHEV PARAMETERS
# ==============================================================================

asinh_term = np.arcsinh(1.0 / epsilon)

alpha = asinh_term / N

sinh_alpha = np.sinh(alpha)

cosh_alpha = np.cosh(alpha)

# ==============================================================================
# STEP 3: COMPUTE STABLE CHEBYSHEV-I POLES FROM THEORY
#
# θk = (2k-1)π/(2N)
#
# pk = -sinh(α) sin(θk)
#      + j cosh(α) cos(θk)
#
# k = 1,...,N
# ==============================================================================

poles_numeric = []

for k in range(1, N + 1):

    theta_k = (2.0 * k - 1.0) * np.pi / (2.0 * N)

    sigma_k = -sinh_alpha * np.sin(theta_k)

    omega_k = cosh_alpha * np.cos(theta_k)

    p_k = sigma_k + 1j * omega_k

    poles_numeric.append(p_k)

poles_numeric = np.array(poles_numeric)

# ==============================================================================
# SYMBOLIC VARIABLES
# ==============================================================================

s = sp.symbols('s', real=True)
omega = sp.symbols('omega', real=True)
I = sp.I

# ==============================================================================
# STEP 4: CONVERT COMPUTED POLES TO SYMBOLIC NUMBERS
#
# The numbers themselves come from the theoretical pole equations above.
# SymPy is only used from this point onward to construct the expressions.
# ==============================================================================

poles_symbolic = []

for p_k in poles_numeric:

    p_sym = sp.Float(p_k.real, 16) + I * sp.Float(p_k.imag, 16)

    poles_symbolic.append(p_sym)

# ==============================================================================
# STEP 5: SYMBOLIC TRANSFER-FUNCTION DENOMINATOR
# ==============================================================================

denominator_symbolic = sp.Integer(1)

for p_k in poles_symbolic:

    denominator_symbolic *= (s - p_k)

denominator_symbolic = sp.expand(denominator_symbolic)

# Remove tiny numerical imaginary residues
denominator_symbolic = sp.N(sp.re(denominator_symbolic), 12)

# ==============================================================================
# STEP 6: NORMALIZATION CONSTANT
#
# N is odd, therefore H(0) = 1.
#
# Hence H0 = D(0).
# ==============================================================================

H0_symbolic = sp.N(denominator_symbolic.subs(s, 0), 12)

H_s = sp.cancel(H0_symbolic / denominator_symbolic)

# ==============================================================================
# STEP 7: POLE-PAIR FACTORS
# ==============================================================================

real_poles = []
upper_poles = []

for p_k in poles_numeric:

    if abs(p_k.imag) < 1e-10:

        real_poles.append(p_k)

    elif p_k.imag > 0:

        upper_poles.append(p_k)

linear_factors = []

for p_k in real_poles:

    factor_k = sp.expand(s - sp.Float(p_k.real, 16))

    linear_factors.append(sp.N(factor_k, 10))

quadratic_factors = []

for p_k in upper_poles:

    sigma = sp.Float(p_k.real, 16)

    omega_k = sp.Float(p_k.imag, 16)

    factor_k = sp.expand(s**2 - 2.0 * sigma * s + sigma**2 + omega_k**2)

    quadratic_factors.append(sp.N(factor_k, 10))

# ==============================================================================
# STEP 8: FREQUENCY RESPONSE
# ==============================================================================

denominator_jw = sp.expand(denominator_symbolic.subs(s, I * omega))

den_real = sp.N(sp.re(denominator_jw), 12)

den_imag = sp.N(sp.im(denominator_jw), 12)

H_jw = sp.cancel(H0_symbolic / denominator_jw)

# ==============================================================================
# STEP 9: MAGNITUDE RESPONSE
# ==============================================================================

magnitude_denominator = sp.expand(den_real**2 + den_imag**2)

magnitude_squared_symbolic = sp.cancel(H0_symbolic**2 / magnitude_denominator)

magnitude_symbolic = sp.sqrt(magnitude_squared_symbolic)

# ==============================================================================
# STEP 10: PHASE RESPONSE
# ==============================================================================

phase_ratio_symbolic = sp.cancel(den_imag / den_real)

# ==============================================================================
# STEP 11: GROUP DELAY
#
# τ(ω) = [R I' - I R'] / [R² + I²]
# ==============================================================================

den_real_derivative = sp.diff(den_real, omega)

den_imag_derivative = sp.diff(den_imag, omega)

group_delay_numerator = sp.expand(den_real * den_imag_derivative - den_imag * den_real_derivative)

group_delay_denominator = sp.expand(den_real**2 + den_imag**2)

group_delay_symbolic = sp.cancel(group_delay_numerator / group_delay_denominator)

# ==============================================================================
# NUMERICAL FUNCTIONS FOR PLOTTING
# ==============================================================================

magnitude_function = sp.lambdify(omega, magnitude_symbolic, 'numpy')

den_real_function = sp.lambdify(omega, den_real, 'numpy')

den_imag_function = sp.lambdify(omega, den_imag, 'numpy')

group_delay_function = sp.lambdify(omega, group_delay_symbolic, 'numpy')

# ==============================================================================
# FREQUENCY AXIS
# ==============================================================================

omega_values = np.logspace(-2, 2, 5000)

# ==============================================================================
# MAGNITUDE VALUES
# ==============================================================================

magnitude_values = np.asarray(magnitude_function(omega_values), dtype=float)

# ==============================================================================
# PHASE VALUES
# ==============================================================================

den_real_values = np.asarray(den_real_function(omega_values), dtype=float)

den_imag_values = np.asarray(den_imag_function(omega_values), dtype=float)

phase_values = -np.unwrap(np.arctan2(den_imag_values, den_real_values))

phase_deg_values = np.rad2deg(phase_values)

# ==============================================================================
# GROUP-DELAY VALUES
# ==============================================================================

group_delay_values = np.asarray(group_delay_function(omega_values), dtype=float)

if group_delay_values.ndim == 0:

    group_delay_values = np.full_like(omega_values, float(group_delay_values))

# ==============================================================================
# TEXT FORMATTING
# ==============================================================================

pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.6f} {p.imag:+.6f}j' for k, p in enumerate(poles_numeric)])

factor_text = ''

for factor_k in linear_factors:

    factor_text += f'Real pole: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

for index, factor_k in enumerate(quadratic_factors):

    factor_text += f'Complex pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

denominator_display = sp.N(denominator_symbolic, 8)

H0_display = float(H0_symbolic)

den_real_display = sp.N(den_real, 8)

den_imag_display = sp.N(den_imag, 8)

magnitude_display = sp.N(magnitude_symbolic, 8)

phase_ratio_display = sp.N(phase_ratio_symbolic, 8)

group_delay_numerator_display = sp.N(group_delay_numerator, 8)

group_delay_denominator_display = sp.N(group_delay_denominator, 8)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1240px;
    max-width:1240px;
    box-sizing:border-box;
">
<b>Chebyshev Type I Design Exercise</b><br>
Construct a normalized Chebyshev Type I low-pass filter with
ω<sub>p</sub> = {wp:.1f} rad/s,
ω<sub>s</sub> = {ws:.1f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.
<br>
<b>Purpose:</b>
Follow the analytical design procedure. All poles and filter coefficients are
computed from the Chebyshev-I theoretical equations; no results from the
printed solution are hard-coded.
</div>
""", layout=Layout(width='1250px', max_width='1250px'))

# ==============================================================================
# INFORMATION PANEL
# ==============================================================================

info_html = HTML(f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    font-size:12px;
    line-height:1.58;
    background:white;
    width:570px;
    box-sizing:border-box;
">

<b>Step 1 — Filter specifications</b><br>
<span style="color:#0066cc;">
ωp = {wp:.1f} rad/s,
ωs = {ws:.1f} rad/s,
Ap = {Ap:.1f} dB,
As = {As:.0f} dB
</span>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 2 — Minimum filter order</b><br>
Nmin = <span style="color:#0066cc;">{N_exact:.6f}</span><br>
N = <span style="color:#0066cc;"><b>{N}</b></span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 3 — Ripple and auxiliary parameters</b><br>
ε = <span style="color:#0066cc;">{epsilon:.6f}</span><br>
sinh⁻¹(1/ε) = <span style="color:#0066cc;">{asinh_term:.6f}</span><br>
α = <span style="color:#0066cc;">{alpha:.6f}</span><br>
sinh(α) = <span style="color:#0066cc;">{sinh_alpha:.6f}</span><br>
cosh(α) = <span style="color:#0066cc;">{cosh_alpha:.6f}</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 4 — Stable poles from the theoretical equations</b><br>
<span style="color:#0066cc;">
{pole_text}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 5 — Pole factors</b><br>
{factor_text}
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 6 — Transfer function</b><br>
H₀ = <span style="color:#0066cc;">{H0_display:.6f}</span><br>
H(s) = H₀ / D(s)<br>
<span style="color:#0066cc;">
D(s) = {sp.sstr(denominator_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 7 — Frequency response denominator</b><br>
D(jω) =
<span style="color:#0066cc;">
({sp.sstr(den_real_display)}) + j({sp.sstr(den_imag_display)})
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 8 — Magnitude response</b><br>
<span style="color:#0066cc;">
|H(jω)| = {sp.sstr(magnitude_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 9 — Phase response</b><br>
∠H(jω) = −tan⁻¹[
<span style="color:#0066cc;">
{sp.sstr(phase_ratio_display)}
</span>
]
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 10 — Group delay</b><br>
τ(ω) =
<span style="color:#0066cc;">
({sp.sstr(group_delay_numerator_display)})
/
({sp.sstr(group_delay_denominator_display)})
</span>
</div>

</div>
""", layout=Layout(width='580px', max_width='580px'))

# ==============================================================================
# COMMON FIGURE SETTINGS
# ==============================================================================

title_fontsize = 11
label_fontsize = 9
tick_fontsize = 8
legend_fontsize = 8

# ==============================================================================
# MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.3, 3.0))

ax_mag.plot(omega_values, magnitude_values, 'r-', linewidth=2.0, label='|H(jω)|')

ax_mag.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_mag.axhline(10.0**(-Ap / 20.0), color='gray', linestyle='--', linewidth=0.9, label='Passband limit')

ax_mag.set_xscale('log')
ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_mag.set_ylabel('|H(jω)|', fontsize=label_fontsize)
ax_mag.set_title('Chebyshev I Magnitude Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_mag.tick_params(axis='both', labelsize=tick_fontsize)
ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)
ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=3, fontsize=legend_fontsize)

ax_mag.set_xlim(0.01, 100.0)
ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False
fig_mag.canvas.layout.width = '530px'
fig_mag.canvas.layout.height = '305px'

# ==============================================================================
# PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.3, 3.0))

ax_phase.plot(omega_values, phase_deg_values, 'r-', linewidth=2.0, label='∠H(jω)')

ax_phase.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')
ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_phase.set_ylabel('Phase (degrees)', fontsize=label_fontsize)
ax_phase.set_title('Chebyshev I Phase Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_phase.tick_params(axis='both', labelsize=tick_fontsize)
ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)
ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_phase.set_xlim(0.01, 100.0)
ax_phase.set_ylim(-455.0, 5.0)
ax_phase.set_yticks([0, -45, -90, -135, -180, -225, -270, -315, -360, -405, -450])

fig_phase.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False
fig_phase.canvas.layout.width = '530px'
fig_phase.canvas.layout.height = '305px'

# ==============================================================================
# GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.3, 3.0))

ax_gd.plot(omega_values, group_delay_values, 'r-', linewidth=2.0, label='τ(ω)')

ax_gd.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xscale('log')
ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_gd.set_ylabel('Group Delay τ(ω)', fontsize=label_fontsize)
ax_gd.set_title('Chebyshev I Group Delay', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=tick_fontsize)
ax_gd.grid(True, which='both', linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_gd.set_xlim(0.01, 100.0)
ax_gd.set_ylim(0.0, 10.0)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '530px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# LAYOUT
# ==============================================================================

left_column = VBox([info_html], layout=Layout(width='590px', min_width='590px', max_width='590px', flex='0 0 590px', align_items='flex-start'))

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_gd.canvas], layout=Layout(width='540px', min_width='540px', max_width='540px', flex='0 0 540px', align_items='flex-start'))

main_layout = HBox([left_column, right_column], layout=Layout(width='1140px', min_width='1140px', max_width='1140px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)